In [1]:
!rm -rf /kaggle/working/pipeline
!git clone https://github.com/matteo-petrelli/Agentic-VQA-Pipeline.git /kaggle/working/pipeline
import sys
sys.path.insert(0, "/kaggle/working/pipeline")



Cloning into '/kaggle/working/pipeline'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 53 (delta 28), reused 47 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 27.95 KiB | 6.99 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [2]:
import subprocess, time

# Installa zstd (richiesto da Ollama) e poi Ollama
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# Avvia il server in background
subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

# Scegli il modello
MODEL = "qwen2.5-vl:3b"
!ollama pull {MODEL}


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
Error: pull model manifest: file does not exist


In [3]:
!pip install -q gliner bitsandbytes nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 83.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-

In [4]:
import sys
sys.path.insert(0, "/kaggle/input/agentic-pipeline")

import config

# Override modello
config.OLLAMA_VLM = MODEL

# Percentuale del dataset (0.1 = 10% per test, 1.0 per run completo)
config.SAMPLING_PERCENTAGE = 0.1

# Verifica path dei dataset
print(f"VLM:       {config.OLLAMA_VLM}")
print(f"Input:     {config.INPUT_JSON_PATH}")
print(f"Images:    {config.IMAGE_DIR}")
print(f"Output:    {config.OUTPUT_JSON_PATH}")
print(f"Sampling:  {config.SAMPLING_PERCENTAGE*100}%")


VLM:       qwen2.5-vl:3b
Input:     /kaggle/input/datasets/matteopetrelli/dude-questions/DUDE_fixed.json
Images:    /kaggle/input/datasets/matteopetrelli/dude-train/content/DUDE_train-val-test_binaries/images/train
Output:    /kaggle/working/agentic_pipeline_results.json
Sampling:  10.0%


In [5]:
import os

# Controlla che i file esistano
assert os.path.exists(config.INPUT_JSON_PATH), f"❌ Input JSON non trovato: {config.INPUT_JSON_PATH}"
assert os.path.isdir(config.IMAGE_DIR), f"❌ Image dir non trovata: {config.IMAGE_DIR}"

# Conta le immagini
imgs = [f for f in os.listdir(config.IMAGE_DIR) if f.endswith(('.jpg', '.png'))]
print(f"✅ Input JSON trovato")
print(f"✅ {len(imgs)} immagini trovate in {config.IMAGE_DIR}")

# Verifica Ollama
import requests
r = requests.get("http://localhost:11434/api/tags")
models = [m["name"] for m in r.json().get("models", [])]
print(f"✅ Ollama attivo, modelli: {models}")


✅ Input JSON trovato
✅ 16894 immagini trovate in /kaggle/input/datasets/matteopetrelli/dude-train/content/DUDE_train-val-test_binaries/images/train
✅ Ollama attivo, modelli: []


In [6]:
from run_experiments import main
main()


=== Agentic VQA Pipeline ===


Sampling enabled: using 18 questions (10.0% of total).
Found 0 already processed questions. Resuming...
[Engine] Initializing Preprocessing Engine...
   - Loading DOTS (Layout/OCR) 4-bit...


config.json: 0.00B [00:00, ?B/s]

configuration_dots.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/strangervisionhf/dots.ocr-base-fix:
- configuration_dots.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


...

  [Error] Processing failed: 404 Client Error: Not Found for url: http://localhost:11434/api/chat
--- [ReAct] Processing: How many Inspector General summary memorandums did state attorneys general obtain for foreclosure-related documents and records?
  [Error] Processing failed: 404 Client Error: Not Found for url: http://localhost:11434/api/chat


Traceback (most recent call last):
  File "/kaggle/working/pipeline/run_experiments.py", line 92, in main
    result = pipeline.process_question(q_text, image_paths)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/pipeline/agentic_pipeline.py", line 42, in process_question
    return self.agent.process_question(question, image_paths)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/pipeline/react_agent.py", line 129, in process_question
    response = self.engine.call_vlm_chat(conversation, primary_image)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/pipeline/preprocessing.py", line 256, in call_vlm_chat
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Error: Not Found for url: http://localhost

--- [ReAct] Processing: If an internee named Jerome is looking to relocate from Santa Barbara during September 1943, where is the best relocation Center?
  [Error] Processing failed: 404 Client Error: Not Found for url: http://localhost:11434/api/chat
--- [ReAct] Processing: How many Inspector General memoranda of review did state attorneys general use to obtain foreclosure-related documents and records?


Traceback (most recent call last):
  File "/kaggle/working/pipeline/run_experiments.py", line 92, in main
    result = pipeline.process_question(q_text, image_paths)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/pipeline/agentic_pipeline.py", line 42, in process_question
    return self.agent.process_question(question, image_paths)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/pipeline/react_agent.py", line 129, in process_question
    response = self.engine.call_vlm_chat(conversation, primary_image)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/pipeline/preprocessing.py", line 256, in call_vlm_chat
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Error: Not Found for url: http://localhost

  [Error] Processing failed: 404 Client Error: Not Found for url: http://localhost:11434/api/chat

=== Finished processing 18 questions ===
Results saved to /kaggle/working/agentic_pipeline_results.json


In [7]:
from evaluate_results import evaluate_results
evaluate_results(config.OUTPUT_JSON_PATH)



 AGENTIC PIPELINE (ReAct) EVALUATION RESULTS

--- ReAct Agent Efficiency ---
Total questions processed : 18
Average steps per question: 0.00
Forced exits (max iters)  : 0 (0.0%)

  Step distribution:
    0 steps:   18 (100.0%) ██████████████████████████████

  Tool usage frequency:

--- QUR (Corrupted Detection Rate) ---
QUR Total : 0.0%  (0/18)
  QUR C1  : 0.0%
  QUR C2  : 0.0%
  QUR C3  : 0.0%

--- FUR (False Unable Rate) ---
FUR Total : 0.0%  (0/0)
  FUR C1  : 0.0%
  FUR C2  : 0.0%
  FUR C3  : 0.0%

--- Overall Metrics ---
Precision : 0.000
Recall    : 0.000
F1 Score  : 0.000

--- Confusion Matrix ---
                     | Agent: 'Unable' | Agent: 'Answer' |
  Actual: Corrupted  | TP: 0           | FN: 18            |
  Actual: Original   | FP: 0           | TN: 0             |


In [8]:
import json

with open(config.OUTPUT_JSON_PATH, "r") as f:
    data = json.load(f)

# Mostra il trace ReAct della prima domanda
q = data["corrupted_questions"][0]
print(f"Domanda: {q['corrupted_question']}")
print(f"Risposta: {q['agentic_result']['final_answer']}")
print(f"Steps: {q['agentic_result']['steps']}")
print(f"Tools: {q['agentic_result']['tools_used']}")
print("\n--- Trace ---")
for step in q['agentic_result']['trace']:
    print(f"\nStep {step['step']}:")
    print(f"  Thought: {step['thought'][:200]}")
    print(f"  Action:  {step['action']}")
    if 'observation' in step:
        print(f"  Observation: {step['observation'][:200]}...")
    if 'answer' in step:
        print(f"  Answer: {step['answer']}")
        print(f"  Confidence: {step['confidence']}")


Domanda: Which timeframe in 2011 had the most people killed in alcohol-impaired driving?
Risposta: Error: 404 Client Error: Not Found for url: http://localhost:11434/api/chat


KeyError: 'steps'